In [1]:
import pandas as pd

In [2]:
# --- 1. Caricamento dei dataset puliti ---
adnimerge = pd.read_csv("ADNIMERGE_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp"])
ptdemog = pd.read_csv("PTDEMOG_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp", "PTDOB"])

In [3]:
adnimerge.dtypes

RID                         int64
COLPROT                    object
VISCODE                    object
EXAMDATE           datetime64[ns]
AGE                       float64
EDUCATION                   int64
APOE4                     float64
CDRSB                     float64
ADAS11                    float64
ADAS13                    float64
MMSE                      float64
RAVLT_immediate           float64
FAQ                       float64
FSVERSION                 float64
IMAGEUID                  float64
Ventricles                float64
Hippocampus               float64
Entorhinal                float64
Fusiform                  float64
MidTemp                   float64
ICV                       float64
update_stamp       datetime64[ns]
GENDER_0                    int64
GENDER_1                    int64
MARRY_0                     int64
MARRY_1                     int64
MARRY_2                     int64
MARRY_3                     int64
ETHNICITY_0                 int64
ETHNICITY_1   

In [ ]:
ptdemog.dtypes

In [4]:
keys = ["RID", "EXAMDATE"]

# Conto delle combinazioni uniche di chiavi
left_keys = adnimerge[keys].drop_duplicates()
right_keys = ptdemog[keys].drop_duplicates()

key_match = left_keys.merge(
    right_keys,
    on=keys,
    how="outer",
    indicator=True
)

counts = key_match["_merge"].value_counts()
print("Conteggio chiavi uniche per [RID, EXAMDATE]:")
print(counts)

print(f"Match: {counts.get('both', 0)}")
print(f"Solo in adnimerge: {counts.get('left_only', 0)}")
print(f"Solo in ptdemog: {counts.get('right_only', 0)}")

Conteggio chiavi uniche per [RID, EXAMDATE]:
_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64
Match: 445
Solo in adnimerge: 8861
Solo in ptdemog: 5515


In [11]:

# --- 2. Merge su RID + EXAMDATE ---
merged = pd.merge(
    adnimerge,
    ptdemog,
    on=keys,
    how="outer",
    indicator=True
)

In [12]:
# --- 3. Log di controllo post-merge ---
print(merged["_merge"].value_counts())
#merged = merged.drop(columns="_merge") #tenere per le visualizazioni eliminare solo alla fine

_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64


In [14]:
merged = merged[sorted(merged.columns)]
merged[merged["_merge"] == "right_only"]

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
0,NaN,NaN,NaN,NaN,60.711841,NaN,NaN,NaN,NaN,NaN,...,NaN,1,NaN,f,NaN,0.0,NaN,right_only,NaT,2005-08-18 00:00:00
1,NaN,NaN,NaN,NaN,74.379192,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,0.0,NaN,right_only,NaT,2005-08-17 00:00:00
3,NaN,NaN,NaN,NaN,79.477070,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,61.0,NaN,right_only,NaT,2013-03-22 15:23:58
4,NaN,NaN,NaN,NaN,80.468172,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,m72,NaN,73.0,NaN,right_only,NaT,2013-05-30 10:05:05
5,NaN,NaN,NaN,NaN,81.297741,NaN,NaN,NaN,NaN,NaN,...,NaN,3,NaN,sc,NaN,0.0,NaN,right_only,NaT,2005-08-18 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.340862,NaN,NaN,NaN,NaN,NaN,...,NaN,10891,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14817,NaN,NaN,NaN,NaN,76.585900,NaN,NaN,NaN,NaN,NaN,...,NaN,10893,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14818,NaN,NaN,NaN,NaN,79.600274,NaN,NaN,NaN,NaN,NaN,...,NaN,10894,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14819,NaN,NaN,NaN,NaN,67.356605,NaN,NaN,NaN,NaN,NaN,...,NaN,10895,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48


Da studiare:
1. per le righe _merge = both --> corrispondono? disugualgianze? origine differenze?
2. capire come unire e togliere _x e _y e avere 1 sola variabile.
3. Che faccio per un soggetto che ha 1 visita in Adnimerge e 3 separate per PTDEMOG? e casi simili? o meglio per una riga solo right o una solo left

In [ ]:
# --- 4. Individuazione righe duplicate sulla chiave, nel risultato finale ---
dup_mask = merged.duplicated(subset=keys, keep=False)   # keep=False: marca TUTTE le occorrenze coinvolte
dup_rows = merged[dup_mask].sort_values(keys)

In [ ]:
print(f"Righe coinvolte in duplicati su {keys}: {dup_mask.sum()} su {len(merged)}")

In [ ]:
# --- 5. Esportazione in CSV ---
merged.to_csv("ADNIMERGE_PTDEMOG_merged_01.csv", index=False, encoding="utf-8-sig")
dup_rows.to_csv("ADNIMERGE_PTDEMOG_duplicati_01.csv", index=False, encoding="utf-8-sig")

In [ ]:
print(f"\nFile salvato: ADNIMERGE_PTDEMOG_merged.csv — {merged.shape[0]} righe, {merged.shape[1]} colonne")
print(f"File duplicati salvato: ADNIMERGE_PTDEMOG_duplicati.csv — {dup_rows.shape[0]} righe")

Controlla PTDOB

In [ ]:
import pandas as pd

df = pd.read_csv("ADNIMERGE_cleaned_02.csv")  # o il nome/percorso corretto del tuo file PTDEMOG

In [ ]:
# --- Ispezione colonna EXAMDATE prima del parsing ---
print("Dtype pandas:", df["EXAMDATE"].dtype)
print("\nPrimi valori non nulli:")
print(df["EXAMDATE"].dropna().unique()[:10])

print("\nType dell'oggetto Python contenuto (primo valore valido):")
primo_valido = df["EXAMDATE"].dropna().iloc[0]
print(type(primo_valido), "->", primo_valido)

print("\nConteggio valori nulli:", df["EXAMDATE"].isna().sum())

In [ ]:
print(df["EXAMDATE"].dropna().unique()[:10])
print(df["update_stamp"].dropna().unique()[:10])